# 00 — Verificación del entorno

Corre este notebook **antes del curso**. Si todas las celdas terminan en ✅ estás listo.

Si algo falla, revisa `alumnos/01_instalacion_y_configuracion.md` (sección *Problemas comunes*).

In [ ]:
import sys, importlib
print("Python:", sys.version.split()[0])
assert sys.version_info >= (3, 11), "Necesitas Python 3.11 o superior (uv lo instala por ti con `uv sync`)."

for paquete in ["numpy", "pandas", "sklearn", "tiktoken", "openai", "dotenv", "matplotlib"]:
    importlib.import_module(paquete)
    print("✅", paquete)

In [ ]:
# Configuración común: localiza el repositorio, carga .env y el documento de trabajo
import os
import json
from pathlib import Path
from dotenv import load_dotenv

def encontrar_raiz():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "data").is_dir() and (c / "pyproject.toml").exists():
            return c
    raise FileNotFoundError("No encontré la carpeta data/. Abre el notebook desde el repositorio del curso.")

ROOT = encontrar_raiz()
load_dotenv(ROOT / ".env")
DATA_DIR = ROOT / "data"

DOCUMENTOS = {
    "cafeteria":   ("cafeteria_nebula_manual.md",      "cafeteria_preguntas.json"),
    "streamly":    ("streamly_api_docs.md",            "streamly_preguntas.json"),
    "veterinaria": ("clinica_veterinaria_huellitas.md", "veterinaria_preguntas.json"),
    "mi_documento": ("mi_documento.md",                 None),  # genera el tuyo con los prompts de alumnos/02_material_del_alumno.md
}

def cargar_documento(nombre):
    archivo, _ = DOCUMENTOS[nombre]
    return (DATA_DIR / archivo).read_text(encoding="utf-8")

def cargar_preguntas(nombre):
    _, archivo = DOCUMENTOS[nombre]
    if archivo is None:
        return []
    return json.loads((DATA_DIR / archivo).read_text(encoding="utf-8"))

print("Repositorio:", ROOT)

In [ ]:
for nombre, (archivo, preguntas) in DOCUMENTOS.items():
    ok = (DATA_DIR / archivo).exists()
    print("✅" if ok else "⚪", archivo, "" if ok else "(opcional: lo crearás tú)")

## Proveedor de LLM

En `.env` define `LLM_PROVEEDOR=openai` (requiere `OPENAI_API_KEY`) o `LLM_PROVEEDOR=ollama`
(requiere tener [Ollama](https://ollama.com/) corriendo y los modelos descargados:
`ollama pull llama3.2` y `ollama pull nomic-embed-text`).

In [ ]:
# Proveedor híbrido: OpenAI (API) u Ollama (local). Se configura en .env con LLM_PROVEEDOR.
# Ollama expone un endpoint compatible con la API de OpenAI, así que usamos el mismo cliente.
import numpy as np
from openai import OpenAI

PROVEEDOR = os.getenv("LLM_PROVEEDOR", "openai").lower()

if PROVEEDOR == "openai":
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Falta OPENAI_API_KEY en .env (o cambia LLM_PROVEEDOR=ollama).")
    client = OpenAI()
    EMBEDDING_MODEL = "text-embedding-3-small"
    CHAT_MODEL = "gpt-4.1-mini"
elif PROVEEDOR == "ollama":
    client = OpenAI(base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"), api_key="ollama")
    EMBEDDING_MODEL = os.getenv("OLLAMA_EMBEDDING_MODEL", "nomic-embed-text")
    CHAT_MODEL = os.getenv("OLLAMA_CHAT_MODEL", "llama3.2")
else:
    raise ValueError("LLM_PROVEEDOR debe ser 'openai' u 'ollama'")

# Nota: los nombres de modelos cambian con el tiempo; verifica la documentación vigente del proveedor.
print(f"Proveedor: {PROVEEDOR} | embeddings: {EMBEDDING_MODEL} | chat: {CHAT_MODEL}")

_cache_embeddings = {}

def get_embeddings(textos, batch_size=64):
    """Devuelve una matriz (n_textos, dimensiones). Cachea resultados para no pagar dos veces."""
    faltantes = [t for t in dict.fromkeys(textos) if t not in _cache_embeddings]
    for i in range(0, len(faltantes), batch_size):
        lote = faltantes[i:i + batch_size]
        resp = client.embeddings.create(model=EMBEDDING_MODEL, input=lote)
        for t, item in zip(lote, resp.data):
            _cache_embeddings[t] = item.embedding
    return np.array([_cache_embeddings[t] for t in textos], dtype=float)

In [ ]:
emb = get_embeddings(["hola mundo"])
print("✅ Embeddings funcionando. Dimensiones:", emb.shape[1])

resp = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": "Responde solo con la palabra: listo"}],
    temperature=0,
)
print("✅ Chat funcionando. Respuesta del modelo:", resp.choices[0].message.content)